# GraceDB BNS/NSBH SESN 3D Crossmatch (Refactor)

*This is a refactor of the original gracedb_sesn_3d_crossmatch.ipynb notebook. It replaces constants and functions previously define within the notebook with calls to modules that do the same.*

This notebook downloads public GraceDB production superevents passing the configured FAR threshold, keeps superevents passing the configured BNS/NSBH probability cut, downloads the best available 3D sky map, and crossmatches stripped-envelope supernovae from TNS.

The workflow is:

1. Download and filter the TNS public catalog for SESN-like types near the configured distance limit.
2. Query GraceDB superevents, fetch `p_astro.json` classifications, and download one best available multiorder FITS skymap per passing superevent.
3. Reproduce the temporal match from `how_many_SESN.ipynb` using the configured discovery-time window.
4. Run a configurable 3D credible-volume crossmatch twice: once with SN luminosity distances from the SHOES cosmology and once with `Planck18`.
5. If any SN lands inside the configured 3D credible volume, save diagnostic overlap plots to `gracedb_sesn_3d_plots/`.


In [2]:
from io import BytesIO
import pandas as pd

## Get transients from TNS

In [3]:
from desi_aap.tns_catalog import download_tns_table, clean_tns_catalog, TNS_CSV_SKIPROWS

In [4]:
tns_data = download_tns_table()
tns_raw = pd.read_csv(BytesIO(tns_data), skiprows=TNS_CSV_SKIPROWS, compression='zip', low_memory=False)
df_sesn = clean_tns_catalog(tns_raw)

df_sesn

,name,ra,declination,redshift,type,discoverydate,reporting_group,internal_names,dist_mpc_SHOES,dist_mpc_Planck18
0,2026peh,300.325188,-20.097515,0.04333,SN Ib,2026-06-12 10:34:32.998000+00:00,ZTF,"ZTF26abbuilh, GOTO26gal, ATLAS26hrq",183.728240,198.266171
1,2026jzl,250.068996,-10.992895,0.06700,SN Ib,2026-04-15 11:04:29.997000+00:00,ZTF,"ZTF26aatbuhc, ATLAS26eqi, GOTO26dyb, PS26djd",288.942282,311.744571
2,2026iqu,159.852142,-11.944218,0.02860,SN Ic,2026-04-07 04:59:05.003000+00:00,ZTF,"ZTF26aarkqap, GOTO26cws, ATLAS26dwc, PS26dfz",119.964004,129.472233
3,2026hmj,193.778789,5.889963,0.04850,SN Ib,2026-03-26 09:09:21.997000+00:00,ZTF,"ZTF26aapyqjm, GOTO26cps, , ATLAS26ebr, PS26del",206.421853,222.745945
4,2026ezk,141.357391,55.594606,0.05800,SN Icn,2026-03-06 04:51:44+00:00,ZTF,"ZTF26aajnagb, GOTO26bsz, ATLAS26ctu, WFST26031...",248.541835,268.175776
...,...,...,...,...,...,...,...,...,...,...
1400,2016Q,122.582750,19.446722,0.10300,SN Ibn,2016-01-07 09:15:50+00:00,Pan-STARRS,PS16hy,455.285325,491.069619
1401,2016P,209.379583,6.097500,0.01460,SN Ic-BL,2016-01-19 04:36:28+00:00,RANSSP,NaN,60.600150,65.410854
1402,2016M,109.157292,67.892306,0.03600,SN IIb,2016-01-15 15:47:00+00:00,NaN,NaN,151.831665,163.855648
1403,2016G,45.990417,43.401000,0.00900,SN Ic-BL,2016-01-09 22:26:24+00:00,NaN,", Gaia16acf",37.197287,40.152037


In [19]:
col_name = "dist_mpc_Planck18"#SHOES"

min_dist = df_sesn[col_name].min()
max_dist = df_sesn[col_name].max()

print("Minimum distance:", min_dist)
print("Maximum distance:", max_dist)

Minimum distance: 0.0
Maximum distance: 537.1370620670143


## Get GraceDB superevents

In [5]:
from desi_aap.gracedb_tools import (
    fetch_gracedb_superevents,
    run_3d_spatial_crossmatch,
    REQUIRE_2D_CREDIBLE_LEVEL,
)

/Users/orl/envs/p312/lib/python3.12/site-packages/ligo/skymap/io/events/ligolw.py:26: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [6]:
gracedb_events = fetch_gracedb_superevents(se_types=["BNS", "NSBH"])
gracedb_events

,superevent_id,gw_time,gps_time,far_hz,far_per_year,p_bns,p_nsbh,p_bbh,p_terrestrial,classification_file,preferred_event,pipeline,search,instruments,labels,skymap_file,skymap_path,status
0,S190425z,2019-04-25 08:18:42.011549+00:00,1.240216e+09,4.537648e-13,1.431973e-05,9.994026e-01,0.000000,0.000000e+00,5.974329e-04,p_astro.json,G330564,gstlal,AllSky,"L1,V1","ADVOK,SKYMAP_READY,EMBRIGHT_READY,PASTRO_READY...",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
1,S190814bv,2019-08-14 21:11:16.012957+00:00,1.249852e+09,2.032625e-33,6.414477e-26,0.000000e+00,0.997890,0.000000e+00,0.000000e+00,p_astro.json,G347305,gstlal,AllSky,"H1,L1,V1","PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_READY,PAS...",bayestar.multiorder.fits,gracedb_skymaps/S190814bv__bayestar.multiorder...,ok
2,S190822c,2019-08-22 01:30:36.589203+00:00,1.250473e+09,6.145185e-18,1.939273e-10,1.000000e+00,0.000000,0.000000e+00,5.223321e-10,p_astro.json,G347846,gstlal,AllSky,"L1,V1","ADVNO,SKYMAP_READY,EMBRIGHT_READY,PASTRO_READY...",bayestar.multiorder.fits,gracedb_skymaps/S190822c__bayestar.multiorder....,ok
3,S190910d,2019-09-10 01:26:56.242676+00:00,1.252114e+09,3.717180e-09,1.173053e-01,0.000000e+00,0.975899,0.000000e+00,2.410074e-02,p_astro.json,G350002,spiir,HighMass,"H1,L1","PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_READY,PAS...",bayestar.multiorder.fits,gracedb_skymaps/S190910d__bayestar.multiorder....,ok
4,S191117j,2019-11-17 06:08:59.454868+00:00,1.258006e+09,1.114482e-18,3.517037e-11,0.000000e+00,1.000000,0.000000e+00,1.080973e-10,p_astro.json,G354833,gstlal,AllSky,"H1,L1","ADVNO,EM_Selected,SKYMAP_READY,EMBRIGHT_READY,...",bayestar.multiorder.fits,gracedb_skymaps/S191117j__bayestar.multiorder....,ok
5,S191205ah,2019-12-05 21:52:45.568738+00:00,1.259618e+09,1.248393e-08,3.939628e-01,0.000000e+00,0.932102,0.000000e+00,6.789774e-02,p_astro.json,G356642,gstlal,AllSky,"H1,L1,V1","EM_READY,ADVOK,EM_Selected,SKYMAP_READY,EMBRIG...",bayestar.multiorder.fits,gracedb_skymaps/S191205ah__bayestar.multiorder...,ok
6,S191220af,2019-12-20 12:24:51.690032+00:00,1.260880e+09,3.963175e-10,1.250683e-02,9.963550e-01,0.000000,0.000000e+00,3.644999e-03,p_astro.json,G357916,gstlal,AllSky,"L1,V1","EM_READY,PE_READY,ADVNO,EM_Selected,SKYMAP_REA...",bayestar.multiorder.fits,gracedb_skymaps/S191220af__bayestar.multiorder...,ok
7,S200116ah,2020-01-16 11:57:19.170712+00:00,1.263211e+09,2.028966e-12,6.402930e-05,0.000000e+00,0.999934,0.000000e+00,6.579270e-05,p_astro.json,G360499,gstlal,AllSky,"H1,L1","EM_READY,PE_READY,ADVNO,EM_Selected,SKYMAP_REA...",bayestar.multiorder.fits,gracedb_skymaps/S200116ah__bayestar.multiorder...,ok
8,S230529ay,2023-05-29 18:15:37.746094+00:00,1.369419e+09,1.975121e-10,6.233008e-03,3.060151e-01,0.624028,0.000000e+00,6.995704e-02,pycbc.p_astro.json,G408702,pycbc,AllSky,L1,"EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",Bilby.multiorder.fits,gracedb_skymaps/S230529ay__Bilby.multiorder.fits,ok
9,S230715bw,2023-07-15 19:06:14.952637+00:00,1.373483e+09,7.843396e-09,2.475188e-01,0.000000e+00,0.906988,8.832113e-02,4.690587e-03,spiir.p_astro.json,G417598,spiir,AllSky,"H1,L1","EM_READY,PE_READY,ADVNO,SKYMAP_READY,EMBRIGHT_...",bayestar.multiorder.fits,gracedb_skymaps/S230715bw__bayestar.multiorder...,ok


## Run temporal crossmatch between transients (SESN) and superevents (GW)

In [6]:
from desi_aap.gracedb_tools import temporal_crossmatch_sesn_to_gw

In [7]:
df_sesn_gracedb_temporal = temporal_crossmatch_sesn_to_gw(df_sesn, gracedb_events)
df_sesn_gracedb_temporal

,name,ra,declination,redshift,type,discoverydate,reporting_group,internal_names,dist_mpc_SHOES,dist_mpc_Planck18,...,gw_p_nsbh,gw_p_bbh,gw_p_terrestrial,gw_preferred_event,gw_pipeline,gw_search,gw_instruments,gw_skymap_file,gw_skymap_path,gw_status
0,2019dgw,287.799129,48.492651,0.095000,SN Ib,2019-04-11 11:47:45+00:00,ZTF,ZTF19aapwnmb,417.673722,450.531339,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
1,2019dps,314.673208,-54.185181,0.044107,SN Ic,2019-04-15 08:23:59+00:00,ASAS-SN,"ASASSN-19kn, Gaia19bky",187.128529,201.934218,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
2,2019dxr,227.863142,5.200552,0.040000,SN Ib,2019-04-15 09:06:36+00:00,ZTF,"ZTF19aarnqys, ATLAS19hxo, PS19wz",169.197122,182.590277,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
3,2019eev,149.274417,8.069490,0.042000,SN IIb,2019-04-21 03:42:38+00:00,ZTF,"ZTF19aarhhfx, ATLAS19jgk, PS19aoi",177.916433,191.996604,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
4,2019dwf,221.131631,70.455981,0.051000,SN IIb,2019-04-21 06:49:26+00:00,ZTF,"ZTF19aarfkch, PS19cxp",217.453479,234.645116,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2025bdy,223.101063,3.716175,0.035700,SN Ib,2025-02-08 11:48:31.003000+00:00,ZTF,"ZTF25aafkxdu, BGEM J145224.28+034258.1, GOTO25...",150.533198,162.454754,...,0.550467,0.0,0.076666,G552809,pycbc,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S250206dm__Bilby.multiorder.fits,ok
174,2025bcn,209.837753,-40.073401,0.013000,SN IIb,2025-02-09 02:08:38.112000+00:00,ATLAS,"ATLAS25bne, GOTO25afo, PS25awq",53.893524,58.172600,...,0.550467,0.0,0.076666,G552809,pycbc,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S250206dm__Bilby.multiorder.fits,ok
175,2025bvu,135.096355,52.064095,0.031647,SN Ib,2025-02-14 09:39:11.808000+00:00,Pan-STARRS,"PS25un, ATLAS25brb, ZTF25aagowuq",133.044973,143.586361,...,0.550467,0.0,0.076666,G552809,pycbc,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S250206dm__Bilby.multiorder.fits,ok
176,2025cav,138.033482,41.317581,0.092000,SN Ic,2025-02-18 22:57:23.328000+00:00,GOTO,"GOTO25amj, ATLAS25btd, PS25yk, WFST0499ckcnz",403.663710,435.429892,...,0.550467,0.0,0.076666,G552809,pycbc,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S250206dm__Bilby.multiorder.fits,ok


In [8]:
### TODO: modularize this too?

temporal_summary = (
    df_sesn_gracedb_temporal.groupby(["superevent_id", "gw_time"], dropna=False)
    .size()
    .rename("n_temporal_sesn")
    .reset_index()
    if not df_sesn_gracedb_temporal.empty
    else pd.DataFrame(columns=["superevent_id", "gw_time", "n_temporal_sesn"])
)

if not gracedb_events.empty:
    gracedb_temporal_summary = gracedb_events.merge(
        temporal_summary, on=["superevent_id", "gw_time"], how="left"
    )
    gracedb_temporal_summary["n_temporal_sesn"] = (
        gracedb_temporal_summary["n_temporal_sesn"].fillna(0).astype(int)
    )
else:
    gracedb_temporal_summary = pd.DataFrame()

display_cols = [
    "superevent_id",
    "gw_time",
    "far_per_year",
    "p_bns",
    "p_nsbh",
    "pipeline",
    "search",
    "skymap_file",
    "n_temporal_sesn",
    "status",
]
gracedb_temporal_summary[[c for c in display_cols if c in gracedb_temporal_summary.columns]]

,superevent_id,gw_time,far_per_year,p_bns,p_nsbh,pipeline,search,skymap_file,n_temporal_sesn,status
0,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,9.994026e-01,0.000000,gstlal,AllSky,GW190425_PublicationSamples.multiorder.fits,13,ok
1,S190814bv,2019-08-14 21:11:16.012957+00:00,6.414477e-26,0.000000e+00,0.997890,gstlal,AllSky,bayestar.multiorder.fits,20,ok
2,S190822c,2019-08-22 01:30:36.589203+00:00,1.939273e-10,1.000000e+00,0.000000,gstlal,AllSky,bayestar.multiorder.fits,20,ok
3,S190910d,2019-09-10 01:26:56.242676+00:00,1.173053e-01,0.000000e+00,0.975899,spiir,HighMass,bayestar.multiorder.fits,13,ok
4,S191117j,2019-11-17 06:08:59.454868+00:00,3.517037e-11,0.000000e+00,1.000000,gstlal,AllSky,bayestar.multiorder.fits,3,ok
5,S191205ah,2019-12-05 21:52:45.568738+00:00,3.939628e-01,0.000000e+00,0.932102,gstlal,AllSky,bayestar.multiorder.fits,13,ok
6,S191220af,2019-12-20 12:24:51.690032+00:00,1.250683e-02,9.963550e-01,0.000000,gstlal,AllSky,bayestar.multiorder.fits,16,ok
7,S200116ah,2020-01-16 11:57:19.170712+00:00,6.402930e-05,0.000000e+00,0.999934,gstlal,AllSky,bayestar.multiorder.fits,20,ok
8,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,3.060151e-01,0.624028,pycbc,AllSky,Bilby.multiorder.fits,10,ok
9,S230715bw,2023-07-15 19:06:14.952637+00:00,2.475188e-01,0.000000e+00,0.906988,spiir,AllSky,bayestar.multiorder.fits,7,ok


**About 2D vs 3D Ranking**

`ligo.skymap.postprocess.crossmatch` reports two different rankings. `searched_prob_2d` is the sky-only credible level after marginalizing over distance. `searched_prob_vol` is a 3D voxel credible level ranked by posterior density per volume at `(RA, Dec, distance)`. These are not nested quantities, so `searched_prob_vol` can be much smaller than `searched_prob_2d` when the SN distance falls on a high-density distance slice even if the sky position is not in the highest-probability sky pixels.

In [9]:
df_sesn_gracedb_3d = run_3d_spatial_crossmatch(df_sesn_gracedb_temporal, gracedb_events)
df_sesn_gracedb_3d

,name,ra,declination,redshift,type,discoverydate,reporting_group,internal_names,dist_mpc_SHOES,dist_mpc_Planck18,...,searched_prob_dist,searched_vol_mpc3,searched_prob_vol,searched_prob_3d_density_rank,probdensity_vol,credible_volume_mpc3,credible_area_deg2,inside_2d_credible_level,inside_3d_credible_level,spatial_status
0,2019dgw,287.799129,48.492651,0.095000,SN Ib,2019-04-11 11:47:45+00:00,ZTF,ZTF19aapwnmb,417.673722,450.531339,...,1.000000,1.661173e+08,1.000004,1.000004,5.397701e-22,2.144799e+06,2400.292101,False,False,ok
1,2019dgw,287.799129,48.492651,0.095000,SN Ib,2019-04-11 11:47:45+00:00,ZTF,ZTF19aapwnmb,417.673722,450.531339,...,0.999999,1.367708e+08,1.000004,1.000004,1.117510e-19,2.144799e+06,2400.292101,False,False,ok
2,2019dps,314.673208,-54.185181,0.044107,SN Ic,2019-04-15 08:23:59+00:00,ASAS-SN,"ASASSN-19kn, Gaia19bky",187.128529,201.934218,...,0.830019,7.130358e+07,1.000004,1.000004,3.358170e-14,2.144799e+06,2400.292101,False,False,ok
3,2019dps,314.673208,-54.185181,0.044107,SN Ic,2019-04-15 08:23:59+00:00,ASAS-SN,"ASASSN-19kn, Gaia19bky",187.128529,201.934218,...,0.741566,5.741121e+07,1.000001,1.000001,5.835956e-13,2.144799e+06,2400.292101,False,False,ok
4,2019dwf,221.131631,70.455981,0.051000,SN IIb,2019-04-21 06:49:26+00:00,ZTF,"ZTF19aarfkch, PS19cxp",217.453479,234.645116,...,0.947668,4.535013e+07,0.999969,0.999969,7.454480e-12,2.144799e+06,2400.292101,False,False,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,2025bvu,135.096355,52.064095,0.031647,SN Ib,2025-02-14 09:39:11.808000+00:00,Pan-STARRS,"PS25un, ATLAS25brb, ZTF25aagowuq",133.044973,143.586361,...,0.022690,2.424860e+09,0.999996,0.999996,1.882578e-257,1.666301e+06,221.882883,False,False,ok
352,2025cav,138.033482,41.317581,0.092000,SN Ic,2025-02-18 22:57:23.328000+00:00,GOTO,"GOTO25amj, ATLAS25btd, PS25yk, WFST0499ckcnz",403.663710,435.429892,...,0.776453,1.106575e+10,0.999996,0.999996,0.000000e+00,1.666301e+06,221.882883,False,False,ok
353,2025cav,138.033482,41.317581,0.092000,SN Ic,2025-02-18 22:57:23.328000+00:00,GOTO,"GOTO25amj, ATLAS25btd, PS25yk, WFST0499ckcnz",403.663710,435.429892,...,0.691387,1.106575e+10,0.999996,0.999996,0.000000e+00,1.666301e+06,221.882883,False,False,ok
354,2025drb,176.154584,1.885731,0.032000,SN Ic,2025-02-19 17:11:02.400000+00:00,WFST,"WFST0521ofmq, PS25aku",134.564118,145.225446,...,0.031100,4.013170e+09,0.999996,0.999996,0.000000e+00,1.666301e+06,221.882883,False,False,ok


In [10]:
### TODO: would like to modularize this one also maybe

if df_sesn_gracedb_3d.empty or not {"spatial_status", "inside_3d_credible_level"}.issubset(
    df_sesn_gracedb_3d.columns
):
    coincidence_sne = pd.DataFrame()
else:
    coincidence_mask = (df_sesn_gracedb_3d["spatial_status"] == "ok") & (
        df_sesn_gracedb_3d["inside_3d_credible_level"] == True
    )
    if REQUIRE_2D_CREDIBLE_LEVEL:
        coincidence_mask &= df_sesn_gracedb_3d["inside_2d_credible_level"] == True
    coincidence_sne = df_sesn_gracedb_3d[coincidence_mask].copy()

coincidence_display_cols = [
    "superevent_id",
    "gw_time",
    "gw_far_per_year",
    "gw_p_bns",
    "gw_p_nsbh",
    "name",
    "type",
    "discoverydate",
    "days_from_gw",
    "redshift",
    "cosmology",
    "sn_dist_mpc",
    "searched_prob_2d",
    "searched_prob_3d_density_rank",
    "searched_prob_dist",
    "searched_area_deg2",
    "credible_area_deg2",
    "credible_volume_mpc3",
    "inside_2d_credible_level",
    "inside_3d_credible_level",
    "ra",
    "declination",
    "reporting_group",
    "internal_names",
]
coincidence_sne[[c for c in coincidence_display_cols if c in coincidence_sne.columns]]

,superevent_id,gw_time,gw_far_per_year,gw_p_bns,gw_p_nsbh,name,type,discoverydate,days_from_gw,redshift,...,searched_prob_dist,searched_area_deg2,credible_area_deg2,credible_volume_mpc3,inside_2d_credible_level,inside_3d_credible_level,ra,declination,reporting_group,internal_names
10,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,0.999403,0.000000,2019ebq,SN Ib/c,2019-04-25 12:11:31+00:00,0.161678,0.037000,...,0.602471,255.564915,2400.292101,2.144799e+06,True,True,255.326411,-7.002923,Pan-STARRS,PS19qp
11,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,0.999403,0.000000,2019ebq,SN Ib/c,2019-04-25 12:11:31+00:00,0.161678,0.037000,...,0.499138,255.564915,2400.292101,2.144799e+06,True,True,255.326411,-7.002923,Pan-STARRS,PS19qp
14,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,0.999403,0.000000,2019eff,SN IIb,2019-04-25 13:59:31+00:00,0.236678,0.050000,...,0.936488,3.147351,2400.292101,2.144799e+06,True,True,248.413109,13.910201,Pan-STARRS,"PS19sh, ZTF19aasckkq"
15,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,0.999403,0.000000,2019eff,SN IIb,2019-04-25 13:59:31+00:00,0.236678,0.050000,...,0.881450,3.147351,2400.292101,2.144799e+06,True,True,248.413109,13.910201,Pan-STARRS,"PS19sh, ZTF19aasckkq"
38,S190814bv,2019-08-14 21:11:16.012957+00:00,6.414477e-26,0.000000,0.997890,2019npv,SN Ib,2019-08-16 07:45:07+00:00,1.440173,0.056000,...,0.374119,4.802989,7.403113,2.628749e+04,True,True,13.384619,-23.832957,DECam-GROWTH,"DG19wxnjc, PS19eqi"
39,S190814bv,2019-08-14 21:11:16.012957+00:00,6.414477e-26,0.000000,0.997890,2019npv,SN Ib,2019-08-16 07:45:07+00:00,1.440173,0.056000,...,0.253252,4.802989,7.403113,2.628749e+04,True,True,13.384619,-23.832957,DECam-GROWTH,"DG19wxnjc, PS19eqi"
236,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,0.306015,0.624028,2023itz,SN Ib,2023-05-16 03:36:13.824000+00:00,-13.610694,0.022000,...,0.041734,25370.168307,8865.930556,1.676365e+07,False,True,327.389230,-15.220088,ATLAS,ATLAS23koz
237,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,0.306015,0.624028,2023itz,SN Ib,2023-05-16 03:36:13.824000+00:00,-13.610694,0.022000,...,0.029054,25370.168307,8865.930556,1.676365e+07,False,True,327.389230,-15.220088,ATLAS,ATLAS23koz
240,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,0.306015,0.624028,2023iwy,SN Ic-BL,2023-05-19 08:23:59+00:00,-10.410865,0.030000,...,0.161019,19209.752952,8865.930556,1.676365e+07,False,True,270.077591,26.408899,ZTF,"ZTF23aakmewi, ATLAS23kzz, PS23dmb"
241,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,0.306015,0.624028,2023iwy,SN Ic-BL,2023-05-19 08:23:59+00:00,-10.410865,0.030000,...,0.119537,19209.752952,8865.930556,1.676365e+07,False,True,270.077591,26.408899,ZTF,"ZTF23aakmewi, ATLAS23kzz, PS23dmb"


## Plot results

In [11]:
from desi_aap.skymap_plots import plot_3d_coincidence

In [12]:
plot_paths = []
if not coincidence_sne.empty:
    for _, row in coincidence_sne.iterrows():
        plot_paths.append(plot_3d_coincidence(row, gracedb_events))

pd.DataFrame({"plot_path": [str(path) for path in plot_paths]})

/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))
/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))
/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))
/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))
/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because fig

,plot_path
0,gracedb_sesn_3d_plots/S190425z__2019ebq__Planc...
1,gracedb_sesn_3d_plots/S190425z__2019ebq__SHOES...
2,gracedb_sesn_3d_plots/S190425z__2019eff__Planc...
3,gracedb_sesn_3d_plots/S190425z__2019eff__SHOES...
4,gracedb_sesn_3d_plots/S190814bv__2019npv__Plan...
5,gracedb_sesn_3d_plots/S190814bv__2019npv__SHOE...
6,gracedb_sesn_3d_plots/S230529ay__2023itz__Plan...
7,gracedb_sesn_3d_plots/S230529ay__2023itz__SHOE...
8,gracedb_sesn_3d_plots/S230529ay__2023iwy__Plan...
9,gracedb_sesn_3d_plots/S230529ay__2023iwy__SHOE...


In [13]:
plot_paths = []
if not coincidence_sne.empty:
    for _, row in coincidence_sne.iterrows():
        plot_paths.append(plot_3d_coincidence(row, gracedb_events))

pd.DataFrame({"plot_path": [str(path) for path in plot_paths]})

/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))
/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))
/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))
/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  f = pylab.figure(fig, figsize=(8.5, 5.4))
/Users/orl/envs/p312/lib/python3.12/site-packages/healpy/visufunc.py:223: UserWarning: Ignoring specified arguments in this call because fig

,plot_path
0,gracedb_sesn_3d_plots/S190425z__2019ebq__Planc...
1,gracedb_sesn_3d_plots/S190425z__2019ebq__SHOES...
2,gracedb_sesn_3d_plots/S190425z__2019eff__Planc...
3,gracedb_sesn_3d_plots/S190425z__2019eff__SHOES...
4,gracedb_sesn_3d_plots/S190814bv__2019npv__Plan...
5,gracedb_sesn_3d_plots/S190814bv__2019npv__SHOE...
6,gracedb_sesn_3d_plots/S230529ay__2023itz__Plan...
7,gracedb_sesn_3d_plots/S230529ay__2023itz__SHOE...
8,gracedb_sesn_3d_plots/S230529ay__2023iwy__Plan...
9,gracedb_sesn_3d_plots/S230529ay__2023iwy__SHOE...
